In [98]:
from langgraph.graph import StateGraph,START,END
from typing import TypedDict,Annotated
from langchain_core.messages import BaseMessage,HumanMessage
from langchain_openai import ChatOpenAI
from langgraph.graph.message import add_messages
from dotenv import load_dotenv

from langgraph.prebuilt import ToolNode,tools_condition
from langchain_core.tools import tool
from langchain_community.tools import DuckDuckGoSearchRun

import requests
import random
import os

In [99]:
load_dotenv()
model = ChatOpenAI()
STOCK_API_KEY = os.getenv("STOCK_API_KEY")

In [100]:
search_tool = DuckDuckGoSearchRun()

@tool
def calculator(query: str) -> str:
    """A calculator tool that evaluates mathematical expressions."""
    try:
        # Evaluate the expression safely
        result = eval(query, {"__builtins__": None}, {})
        return str(result)
    except Exception as e:
        return f"Error evaluating expression: {e}"
    

@tool
def get_stock_price(ticker: str) -> str:
    """A tool that fetches the current stock price for a given ticker symbol."""
    try:
        response = requests.get(f"https://finnhub.io/api/v1/quote?symbol={ticker}&token={STOCK_API_KEY}")
        data = response.json()
        if "c" in data:
            return f"The current stock price of {ticker} is ${data['c']}"
        else:
            return f"Could not retrieve stock price for {ticker}."
    except Exception as e:
        return f"Error fetching stock price: {e}"                                                      

In [101]:
tools = [calculator,get_stock_price,search_tool]

models_with_tools =model.bind_tools(tools)

In [102]:
class ChatState(TypedDict):
    messages: Annotated[list[BaseMessage],add_messages]

In [103]:
def chat_node(state: ChatState):
    """LLM node that takes in a list of messages and returns a response."""
    messages = state["messages"]
    response = models_with_tools.invoke(messages)
    return {"messages": response}

tool_node=ToolNode(tools=tools)

In [104]:
graph = StateGraph(ChatState)
graph.add_node('chat', chat_node)
graph.add_node('tools', tool_node)


In [105]:
graph.add_edge(START,"chat")

graph.add_conditional_edges('chat',tools_condition)
graph.add_edge('tools','chat')
chatbot = graph.compile()



In [107]:
result = chatbot.invoke({"messages":[HumanMessage(content="Who is the CEO and CTO of apple and its current stock price of AAPL?")]})
print(result["messages"][-1].content)

The current CEO of Apple is Tim Cook, and the current CTO of Apple is not specifically mentioned in the search results. The current stock price of AAPL is $310.66.
